In [0]:
%sql
SELECT DATE_TRUNC('HOUR', session_start), created_at, COUNT(*)
FROM prod.detection.viewing_content_firehose
WHERE tuner_channel_id IS NOT NULL
  AND partition_key = '2026-06-08'
GROUP BY 1, 2

In [0]:
%sql
SELECT DATE_TRUNC('HOUR', session_start), created_at, COUNT(*)
FROM prod.detection.viewing_commercials_firehose
WHERE tuner_channel_number IS NOT NULL
  AND partition_key = '2026-06-08'
GROUP BY 1, 2 

In [0]:
from pyspark.sql.functions import col, date_trunc, hour

# Define the hours of interest
hours_of_interest = ["2026-06-08 18:00:00", "2026-06-09 13:00:00"]

# Load relevant columns from both tables for the specified hours
content_df = spark.table("prod.detection.viewing_content_firehose") \
    .filter((col("partition_key").isin("2026-06-08", "2026-06-09")) &
            (date_trunc("hour", col("session_start")).cast("string").isin(hours_of_interest))) \
    .select("session_start", "created_at", "tuner_channel_id", "partition_key")

commercials_df = spark.table("prod.detection.viewing_commercials_firehose") \
    .filter((col("partition_key").isin("2026-06-08", "2026-06-09")) &
            (date_trunc("hour", col("session_start")).cast("string").isin(hours_of_interest))) \
    .select("session_start", "created_at", "tuner_channel_number", "partition_key")

# Join on session_start and created_at to compare tuner columns
joined_df = content_df.join(
    commercials_df,
    ["session_start", "created_at", "partition_key"],
    "left"
).filter(
    (col("tuner_channel_id").isNotNull()) &
    ((col("tuner_channel_number").isNull()) | (col("tuner_channel_id") != col("tuner_channel_number")))
)

display(joined_df)

In [0]:
# Count the number of mismatched or missing tuner_channel_number records per hour
result_df = joined_df.withColumn("hour", date_trunc("hour", col("session_start"))) \
    .groupBy("hour", "partition_key") \
    .count() \
    .orderBy("hour", "partition_key")

display(result_df)

In [0]:
# Display records from content_df where there is no matching tuner_channel_number in commercials_df
missing_data_df = joined_df.filter(col("tuner_channel_number").isNull())
display(missing_data_df)

In [0]:
from pyspark.sql import functions as F

AFFECTED_HOURS = ["2026-06-08 18:00:00", "2026-06-09 13:00:00"]
AFFECTED_PARTITIONS = ["2026-06-08", "2026-06-09"]

comm = spark.table("prod.detection.viewing_commercials_firehose")
cont = spark.table("prod.detection.viewing_content_firehose")

# ── 1. Are there ANY rows in commercials for those hours (regardless of tuner)? ──
print("=" * 70)
print("1. Total rows in viewing_commercials_firehose for the affected hours")
print("   (no tuner filter — any row counts)")
print("=" * 70)
comm_any = (
    comm
    .filter(F.col("partition_key").isin(AFFECTED_PARTITIONS))
    .filter(F.date_trunc("hour", F.col("session_start")).cast("string").isin(AFFECTED_HOURS))
    .agg(F.count("*").alias("total_rows"))
)
comm_any.show()

# ── 2. Same hours in content table (baseline expected volume) ──
print("=" * 70)
print("2. Total rows in viewing_content_firehose for the affected hours (baseline)")
print("=" * 70)
cont_any = (
    cont
    .filter(F.col("partition_key").isin(AFFECTED_PARTITIONS))
    .filter(F.date_trunc("hour", F.col("session_start")).cast("string").isin(AFFECTED_HOURS))
    .agg(F.count("*").alias("total_rows"))
)
cont_any.show()

# ── 3. Null rates across ALL columns in commercials for the affected hours ──
#    If rows exist but multiple columns are null → ingestion wrote empty records.
#    If zero rows → batch was never written.
print("=" * 70)
print("3. Null rates for ALL columns in commercials for affected hours")
print("   (helps distinguish: rows absent vs rows present but all-null)")
print("=" * 70)
comm_affected = (
    comm
    .filter(F.col("partition_key").isin(AFFECTED_PARTITIONS))
    .filter(F.date_trunc("hour", F.col("session_start")).cast("string").isin(AFFECTED_HOURS))
)
total = comm_affected.count()
if total == 0:
    print(f"  >>> ZERO rows found in commercials for affected hours — batch was never written <<<")
else:
    print(f"  Total rows found: {total:,}")
    null_counts = comm_affected.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in comm_affected.columns]
    ).collect()[0].asDict()
    print("  Null counts per column:")
    for col_name, null_count in sorted(null_counts.items(), key=lambda x: -x[1]):
        pct = null_count / total * 100
        print(f"    {col_name:<40} {null_count:>10,}  ({pct:.1f}%)")

# ── 4. created_at batch timestamps in content vs commercials around the gap ──
#    Each hour's batch has the same created_at. Compare which created_at values
#    exist in content but NOT in commercials (missing ingestion runs).
print("\n" + "=" * 70)
print("4. created_at batch timestamps present in content but ABSENT in commercials")
print("   for partition_key 2026-06-08 (checks if the pipeline run was skipped)")
print("=" * 70)
content_batches = (
    cont
    .filter(F.col("partition_key") == "2026-06-08")
    .select(F.date_trunc("hour", F.col("session_start")).alias("hour"),
            F.col("created_at"))
    .distinct()
)
comm_batches = (
    comm
    .filter(F.col("partition_key") == "2026-06-08")
    .select(F.date_trunc("hour", F.col("session_start")).alias("hour"),
            F.col("created_at"))
    .distinct()
)
batch_diff = (
    content_batches.alias("c")
    .join(comm_batches.alias("m"),
          (F.col("c.hour") == F.col("m.hour")) & (F.col("c.created_at") == F.col("m.created_at")),
          "left")
    .filter(F.col("m.hour").isNull())
    .select(F.col("c.hour"), F.col("c.created_at"))
    .orderBy("c.hour")
)
print("  Hours in content with NO matching batch in commercials (same created_at):")
batch_diff.show(50, truncate=False)

# ── 5. For the affected content batches: what created_at do they carry? ──
print("=" * 70)
print("5. created_at of the affected content batches (the runs that should have")
print("   populated commercials but didn't)")
print("=" * 70)
affected_batch_created_at = (
    cont
    .filter(F.col("partition_key").isin(AFFECTED_PARTITIONS))
    .filter(F.date_trunc("hour", F.col("session_start")).cast("string").isin(AFFECTED_HOURS))
    .select("partition_key",
            F.date_trunc("hour", F.col("session_start")).alias("hour"),
            "created_at")
    .distinct()
    .orderBy("partition_key", "hour")
)
affected_batch_created_at.show(truncate=False)

# ── 6. Check if those created_at values appear ANYWHERE in commercials ──
#    (rules out the case where commercials landed on a different partition_key)
print("=" * 70)
print("6. Do those specific created_at values appear anywhere in commercials?")
print("   (checks if data landed on a different partition)")
print("=" * 70)
batch_rows = affected_batch_created_at.collect()
for row in batch_rows:
    ca = row["created_at"]
    cnt = comm.filter(F.col("created_at") == ca).count()
    print(f"  created_at={ca}  →  {cnt:,} rows in commercials anywhere")

In [0]:
from pyspark.sql import functions as F

comm = spark.table("prod.detection.viewing_commercials_firehose")

AFFECTED_HOURS   = ["2026-06-08 18:00:00", "2026-06-09 13:00:00"]
AFFECTED_PARTS   = ["2026-06-08", "2026-06-09"]
# Use a neighbouring healthy hour as reference (17:00 on 2026-06-08 is confirmed present in Cell 2)
REF_HOUR         = "2026-06-08 17:00:00"
REF_PART         = "2026-06-08"

def null_pct(df):
    n = df.count()
    if n == 0:
        return {}, 0
    counts = df.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    ).collect()[0].asDict()
    return {k: round(v / n * 100, 1) for k, v in counts.items()}, n

df_affected = (
    comm
    .filter(F.col("partition_key").isin(AFFECTED_PARTS))
    .filter(F.date_trunc("hour", F.col("session_start")).cast("string").isin(AFFECTED_HOURS))
)
df_ref = (
    comm
    .filter(F.col("partition_key") == REF_PART)
    .filter(F.date_trunc("hour", F.col("session_start")).cast("string") == REF_HOUR)
)

pct_affected, n_affected = null_pct(df_affected)
pct_ref,      n_ref      = null_pct(df_ref)

print(f"Reference hour ({REF_HOUR}):         {n_ref:>15,.0f} rows")
print(f"Affected hours ({AFFECTED_HOURS}): {n_affected:>15,.0f} rows")
print()
print(f"{'Column':<42}  {'Ref null%':>10}  {'Affected null%':>14}  Status")
print("-" * 82)

missing_in_affected = []
for col_name in sorted(comm.columns):
    ref_p  = pct_ref.get(col_name,  0.0)
    aff_p  = pct_affected.get(col_name, 0.0)
    # Flag columns that are healthy in reference but fully null in affected
    if aff_p == 100.0 and ref_p < 5.0:
        status = "<-- MISSING (backfill needed)"
        missing_in_affected.append(col_name)
    elif aff_p > ref_p + 5:
        status = "<-- elevated nulls"
    else:
        status = ""
    print(f"  {col_name:<40}  {ref_p:>10.1f}  {aff_p:>14.1f}  {status}")

print()
print("=" * 82)
print(f"Columns requiring backfill ({len(missing_in_affected)} total):")
for c in missing_in_affected:
    print(f"  - {c}")

In [0]:
%sql
SELECT created_at, MIN(ts_start), MAX(ts_end), COUNT(*) FROM prod.cooker.vizio_attrcomm_firehose
WHERE created_at = '2026-06-09T16:45:25.539'
--   AND created_at < '2026-06-08T22:00:00'
-- WHERE ts_start >= '2026-06-09 13:00:00'
--   AND ts_start < '2026-06-09 14:00:00'
GROUP BY 1

In [0]:
%sql
SELECT COUNT(1) FROM
     (WITH vc_start AS (
          SELECT MIN(ts_start) AS min_start,MAX(ts_end) AS max_end
          FROM prod.cooker.vizio_attrcomm_firehose
          WHERE created_at = '2026-06-09T16:45:25.539'
     )                           
     select 
          fk_tvid
          ,session_start
          ,session_end
          ,is_live
          ,if(vizio_epg_station IS NOT NULL,NULL,fk_content_id) fk_content_id
          ,if(vizio_epg_station IS NOT NULL,NULL,fk_show_id) fk_show_id
          ,if(vizio_epg_station IS NOT NULL,NULL,fk_station_id) fk_station_id
          ,vizio_epg_station
          ,vizio_epg_program
          ,tuner_channel_id
          ,tuner_program_id
          ,if(vizio_epg_station IS NOT NULL,NULL,tms_show_id) tms_show_id
          ,if(vizio_epg_station IS NOT NULL,NULL,tms_station_id) tms_station_id
          ,tms_tuner_channel_id
          ,tms_tuner_program_id
          ,tuner_channel_number
          ,content_type
     from prod.detection.viewing_content_firehose
     where session_start >= date_trunc('hour', (SELECT min_start FROM vc_start))
     and session_start < date_trunc('hour', (SELECT max_end + interval '1 hour' FROM vc_start))
     and session_start <> session_end)